## Example: Using the ecFactory workflow engine for strain design

In [1]:
import sys
import copy
sys.path.append('..')
import cobra
from strainOptimizer.strainDesign.workflow_engine import (
    strainOptimizer_engine,
    WorkflowParameters,
)

# Set tolerance
cobra.Configuration().tolerance = 1e-9

Set parameter LicenseID to value 2624035


### (Optional) Add heterologous pathway into model
If your target product is heterologous, you can add the heterologous pathway into the model. Please refer to the example [add_sclareol_pathway.py](0.add_sclareol_pathway.py)

### 1. Prepare parameters

StrainOptimizer provides a class `WorkflowParameters` to store the parameters for the workflow. It includes three types of parameters: `model`, `strain`, and `algorithm`.
- model: Model-related parameters
    - model_path: The path to the model file
    - model_type: The type of the model, e.g., 'ecGEM', 'etfl', 'gem', 'GAN_ec'
    - solver: The solver to be used
    - growth_id: The growth reaction id
- strain: Strain-related parameters
    - target_id: The target reaction id
    - product_name: The name of the product
    - c_source: The carbon source reaction id
    - c_uptake: The carbon source uptake rate
- algorithm: Algorithm-related parameters
    - design_algorithm: The design algorithm to be used. Currently, the supported algorithms are 'ecFactory', 'iBridge', 'ecFSEOF'.
    - simulation_method: The simulation method to be used. Currently, the supported methods are 'ppfba', 'pfba', 'moma', 'mopa'.
    - remove_essential: Whether to remove the essential reactions.
    - output_directory: The directory to save the results.
    - save_results: Whether to save the results.
    - experimental_yield: The experimental yield of the product.
    - scanning_range: The range of the scanning.
    

In [7]:
model_params = {
    'model_path': 'models\yeast\ecYeastGEM_batch.xml',
    'model_type': 'ecGEM',
    'solver': 'optlang-gurobi',
    'growth_id': 'r_2111',
}


# Strain parameters - target product and growth conditions
strain_params = {
    'target_id': 'r_1589',
    'product_name': '2-phenylethanol',
    'c_source': 'r_1714_REV',  # glucose exchange reaction
    'c_uptake': 5,  # glucose uptake rate (mmol/gDW/h)
}

# Algorithm control parameters - workflow and output settings
algorithm_params = {
    'design_algorithm': 'ecFactory',
    'simulation_method': 'ppfba',
    'experimental_yield': None, # if without experimental yield data, use the 1/2 
    'remove_essential': True,
    'output_directory': './results',
    'steps':123,
    'action_thresholds':[0.05,0.3,1.1]
    # 'save_results': False,
    # 'only_final_result': True,
    # Note: ecFactory-specific parameters like steps, action_thresholds, etc.
    # would need to be added to AlgorithmControl if they're used

}

# Create WorkflowParameters using the three-level structure
params = WorkflowParameters(
    model=model_params,
    strain=strain_params,
    algorithm=algorithm_params
)

### 2. Run the ecFactory design workflow for ecModel

In [3]:
# Create workflow engine using the new framework
engine = strainOptimizer_engine(params)
print(f"Engine created for {params.strain['product_name']} cell factory design")
print(f"Target reaction: {params.strain['target_id']}")
print(f"Carbon source: {params.strain['c_source']}")
print(f"Model type: {params.model['model_type']}")
print(f"Algorithm: {params.algorithm['design_algorithm']}")

# Load model
model=engine.load_model()

Parameters validated successfully
Engine created for 2-phenylethanol cell factory design
Target reaction: r_1589
Carbon source: r_1714_REV
Model type: ecGEM
Algorithm: ecFactory
Loading ecGEM model from models\yeast\ecYeastGEM_batch.xml


In [ ]:
# Run the design workflow
combined_result = engine.run_design()
result_level1=engine.all_results['level 1 result']
result_level2=engine.all_results['level 2 result']
# result_level3=

Starting strain design workflow using ecFactory
Running ecFactory algorithm
Read LP format model from file C:\Users\wangh\AppData\Local\Temp\tmpx3hlbgsa.lp
Reading time = 0.03 seconds
: 4180 rows, 16288 columns, 60346 nonzeros
1.-  **** Running ecFSEOF ****
when growth rate is 0.033088,production rate is 2.575721829162254
when growth rate is 0.070903,production rate is 2.3434883734281997
when growth rate is 0.108718,production rate is 2.111254917694147
when growth rate is 0.146532,production rate is 1.8790276032652917
when growth rate is 0.184347,production rate is 1.646794147531224
when growth rate is 0.222162,production rate is 1.3815486299479178
when growth rate is 0.259977,production rate is 0.937301736959987
when growth rate is 0.297792,production rate is 0.4621253879155333
Found 4968 reactions with all NaN values
Found 91 reactions with inconsistent patterns
inconsistent k_scores for gene YHR208W
inconsistent k_scores for gene YJR148W
inconsistent k_scores for gene YLL052C
incons

..\strainOptimizer\strainDesign\workflow_engine.py:199: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results['level 2 result']['gene_name'] = gene_id_to_name(results['level 2 result'].index)
..\strainOptimizer\strainDesign\workflow_engine.py:200: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results['level 3 result']['gene_name'] = gene_id_to_name(results['level 3 result'].index)


In [6]:
result

,gene_name,k_score,action,EUVA_priority_level,minimal_candidates_set
YOL086C,ADH1,1000.0,OE,3.0,0.0
YDR380W,ARO10,1000.0,OE,1.0,1.0
YDL168W,SFA1,1000.0,OE,discarded by EUVA,0.0
YBR145W,ADH5,1000.0,OE,discarded by EUVA,0.0
YGR177C,ATF2,0.0,KO,3.0,0.0
YOR377W,ATF1,0.0,KO,3.0,0.0


### 3. Run the ecFactory design workflow for ETFL model

In [10]:
efl_model_params = {
    'model_path': 'models\yeast\yeast8_cEFL_2584_enz_64_bins__20231221_083715.json',
    'model_type': 'etfl',
    'solver': 'optlang-gurobi',
    'growth_id': 'r_2111',
    'c_source': 'r_1714',
    'c_uptake': 1,
}

engine.update_parameters(update_dict=efl_model_params)

print(f"Engine created for {params.strain['product_name']} cell factory design")
print(f"Target reaction: {params.strain['target_id']}")
print(f"Carbon source: {params.strain['c_source']}")
print(f"Model type: {params.model['model_type']}")
print(f"Algorithm: {params.algorithm['design_algorithm']}")

Updated parameter model.model_path = models\yeast\yeast8_cEFL_2584_enz_64_bins__20231221_083715.json
Updated parameter model.model_type = etfl
Updated parameter model.solver = optlang-gurobi
Updated parameter model.growth_id = r_2111
Updated parameter strain.c_source = r_1714
Updated parameter strain.c_uptake = 1
Parameters validated successfully
Loading etfl model from models\yeast\yeast8_cEFL_2584_enz_64_bins__20231221_083715.json
Read LP format model from file C:\Users\wangh\AppData\Local\Temp\tmp3olo4rla.lp
Reading time = 0.08 seconds
: 4059 rows, 20592 columns, 125006 nonzeros


2025-12-01 09:20:07,026 - ME modelyeast8_cEFL_2584_enz_64_bins__20231221_083715 - INFO - # ETFL Model ETFL_model initialized
2025-12-01 09:20:07,026 - ME modelyeast8_cEFL_2584_enz_64_bins__20231221_083715 - INFO - Empty model initialized
rebuilding constraints: 100%|██████████| 63519/63519 [00:12<00:00, 5138.55it/s]


Engine created for 2-phenylethanol cell factory design
Target reaction: r_1589
Carbon source: r_1714_REV
Model type: ecGEM
Algorithm: ecFactory


In [11]:
# Run the design workflow
result = engine.run_design()

Starting strain design workflow using ecFactory
Running ecFactory algorithm
Read LP format model from file C:\Users\wangh\AppData\Local\Temp\tmpkj1c0yi2.lp
Reading time = 0.08 seconds
: 4059 rows, 20592 columns, 125006 nonzeros


2025-12-01 09:25:10,448 - ME modelyeast8_cEFL_2584_enz_64_bins__20231221_083715 - INFO - # ETFL Model ETFL_model initialized
2025-12-01 09:25:10,449 - ME modelyeast8_cEFL_2584_enz_64_bins__20231221_083715 - INFO - Empty model initialized
rebuilding constraints: 100%|██████████| 63519/63519 [00:08<00:00, 7059.42it/s]


1.-  **** Running ecFSEOF ****
when growth rate is 0.008147,production rate is 0.4499570077382632


2025-12-01 09:30:21,814 - ME modelyeast8_cEFL_2584_enz_64_bins__20231221_083715 - WARNING - Exception occurred during solving: Unable to retrieve attribute 'X'. - Solver status: infeasible


Workflow failed: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices


IndexError: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices